In [ ]:
!pip install --upgrade --index-url https://pypi.ngc.nvidia.com nvidia-tensorrt

In [ ]:
!sudo apt-get install -y tensorrt

In [ ]:
import tensorrt
print(tensorrt.__version__)
assert tensorrt.Builder(tensorrt.Logger())

In [ ]:
!sudo apt-get install python3-libnvinfer-dev -y
!python3 -m pip install numpy onnx
!sudo apt-get install onnx-graphsurgeon -y
#--------------------------------------------------
#upgrade pytorch to 1.12
#!pip install torch==1.12.1+cu116 torchvision==0.13.1+cu116 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu116

In [ ]:
!dpkg-query -W tensorrt
!dpkg -l | grep nvinfer

In [ ]:
!pip install --upgrade pip
!pip install --upgrade sphinx-glpi-theme
!pip install --no-cache-dir --index-url https://pypi.nvidia.com pytorch-quantization
!pip install wget
!pip3 install -U pip && pip3 install onnxsim
!pip install onnxruntime

# Import libraries and benchmarking functions

In [3]:
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision
import torchvision.transforms as transforms
from torchvision import models, datasets
from torch.utils.tensorboard import SummaryWriter
import onnxruntime
import onnxsim


import pytorch_quantization
from pytorch_quantization import nn as quant_nn
from pytorch_quantization import quant_modules
from pytorch_quantization import calib
from pytorch_quantization.tensor_quant import QuantDescriptor
from pytorch_quantization.nn.modules.tensor_quantizer import TensorQuantizer
from pytorch_quantization.nn.modules import quant_pooling
from tqdm import tqdm

print(pytorch_quantization.__version__)
# print(torch_tensorrt.__version__)
print(onnxruntime.__version__)
print(onnxsim.__version__)
#print(torchaudio.__version__)

import os
import sys
import warnings
import time
import numpy as np
import wget
import tarfile
import shutil
warnings.simplefilter('ignore')
import collections
from absl import logging
logging.set_verbosity(logging.FATAL)  # Disable logging as they are too noisy in notebook

import json


!mkdir -p /root/.cache/torch/hub/checkpoints/
!cp /kaggle/input/few-imagenet-examples/efficientnet_b0_ra-3dd342df.pth /root/.cache/torch/hub/checkpoints/
!cp /kaggle/input/few-imagenet-examples/efficientnet_b2_ra-bcdf34b7.pth /root/.cache/torch/hub/checkpoints/

# loading labels
with open("/kaggle/input/few-imagenet-examples/data/imagenet_class_index.json") as json_file: 
    d = json.load(json_file)


2.1.3
1.14.1
0.4.36


In [ ]:
import os
import pandas as pd
from shutil import copyfile

# Define root directory and output path
root_dir = "/kaggle/input/imagenet/mini-imagenet"
output_dir = "/kaggle/working/mini-imagenet"

# Create output directories
splits = ['train', 'val', 'test']
for split in splits:
    os.makedirs(os.path.join(output_dir, split), exist_ok=True)

# Function to process and copy images
def process_split(csv_file, image_dir, split):
    df = pd.read_csv(csv_file)
    for _, row in df.iterrows():
        label_dir = os.path.join(output_dir, split, row['label'])
        os.makedirs(label_dir, exist_ok=True)
        
        # Construct source and destination paths
        source = os.path.join(image_dir, row['label'], row['filename'])  # Include label as subfolder
        destination = os.path.join(label_dir, row['filename'])
        
        # Check if file exists and copy it
        if os.path.exists(source):
            copyfile(source, destination)
        else:
            print(f"Warning: {source} does not exist! Skipping...")

# Paths to CSV files
csv_dir = "/kaggle/input/imagenet/mini-imagenet"  # Directory where CSV files are located

# Process each split (train, val, test)
process_split(os.path.join(csv_dir, "train.csv"), os.path.join(root_dir, "train"), "train")
process_split(os.path.join(csv_dir, "val.csv"), os.path.join(root_dir, "val"), "val")
process_split(os.path.join(csv_dir, "test.csv"), os.path.join(root_dir, "test"), "test")

print("Preprocessing completed successfully!")


In [ ]:
!tar xzvf /kaggle/input/imagenette/imagenette2-320.tgz

In [4]:
# Define main data directory
DATA_DIR = '/kaggle/working/imagenette2-320' 
# Define training and validation data paths
TRAIN_DIR = os.path.join(DATA_DIR, 'train') 
VAL_DIR = os.path.join(DATA_DIR, 'val')


# **Data Pre-processing**

In [5]:
#Performing Transformations on the dataset and defining training and validation dataloaders
# Define data transforms
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),  # Match validation
        # transforms.RandomHorizontalFlip(),  # Minimal augmentation
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),
}

# data_dir = '/kaggle/working/mini-imagenet'
train_dataset = ImageFolder(os.path.join(DATA_DIR, 'train'), transform=data_transforms['train'])
val_dataset = ImageFolder(os.path.join(DATA_DIR, 'val'), transform=data_transforms['val'])

train_dataloader = data.DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True,pin_memory=True,num_workers=4)
val_dataloader = data.DataLoader(val_dataset, batch_size=32, shuffle=False, drop_last=False,pin_memory=True,num_workers=4)
# calib_dataloader = data.DataLoader(calib_dataset, batch_size=32, shuffle=False, drop_last=True,pin_memory=True,num_workers=4)


#Data Augmentation can be done

In [6]:
#This function allows you to set the all the parameters to not have gradients, 
#allowing you to freeze the model and not undergo training during the train step. 
def set_parameter_requires_grad(model, feature_extracting):
    if feature_extracting:
        for param in model.parameters():
            param.requires_grad = False

feature_extract = True #This varaible can be set False if you want to finetune the model by updating all the parameters. 
#--------Load Torchvision Model------------------
model = models.mobilenet_v2(pretrained=False)
# model = models.resnet50(pretrained=True)
set_parameter_requires_grad(model, feature_extract)
#Define a classification head for 10 classes.
#-----Mobilenetv2---------------------------
model.classifier[1] = nn.Linear(1280, 10)
# Change dropout probability to 0.5
# model.classifier[0] = nn.Dropout(p=0.2, inplace=False)
# model.classifier[0] = nn.Identity()

#-----Resnet50------------------------------
# num_ftrs = model.fc.in_features
# model.fc = nn.Linear(num_ftrs, 10)
#-------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.cuda()

In [8]:
# mobilenetv2_base_ckpt is the checkpoint generated from Step 2 : Training a baseline Mobilenetv2 model.
ckpt = torch.load("/kaggle/working/best_mobilenetv2_ckpt")
# ckpt = torch.load("/kaggle/working/resnet50_base_ckpt")
modified_state_dict={}
for key, val in ckpt["model_state_dict"].items():
    # Remove 'module.' from the key names
    if key.startswith('module'):
        modified_state_dict[key[7:]] = val
    else:
        modified_state_dict[key] = val

# Load the pre-trained checkpoint
model.load_state_dict(modified_state_dict)
optimizer.load_state_dict(ckpt["opt_state_dict"])

In [7]:
# Declare Learning rate
from torch.optim.lr_scheduler import StepLR,ReduceLROnPlateau
#lr = 0.0001 #good enough for resnet50
lr = 0.001

# Use cross entropy loss for classification and SGD optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.classifier[1].parameters(), lr=1e-3, momentum=0.9)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3, verbose=True)



In [9]:
# Define functions for training, evalution, saving checkpoint and train parameter setting function
def train(model, dataloader, crit, opt, epoch):
    model.train()
    total_loss = 0.0
    total_corrects = 0
    total_samples = 0

    for data, labels in dataloader:
        data, labels = data.to(device), labels.to(device)

        # Zero the parameter gradients
        opt.zero_grad()

        # Forward pass
        out = model(data)
        loss = crit(out, labels)

        # Backward pass and optimize
        loss.backward()
        opt.step()

        # Accumulate loss
        total_loss += loss.item()

        # Calculate number of correct predictions
        _, preds = torch.max(out, 1)
        total_corrects += (preds == labels).sum().item()
        total_samples += labels.size(0)

    # Calculate average loss and accuracy for the epoch
    avg_loss = total_loss / total_samples
    avg_acc = total_corrects / total_samples

    # print(f"Epoch [{epoch}] - Loss: {avg_loss:.4f}, Accuracy: {avg_acc * 100:.2f}%")
    return avg_loss, avg_acc

    

def evaluate(model, dataloader, crit, epoch, dtype='fp32'):
    model.eval()
    total_loss = 0.0
    total_corrects = 0
    total_samples = 0

    with torch.no_grad():  # Disable gradient computation
        for data, labels in dataloader:
            # Ensure data type consistency
            if dtype == 'fp16':
                data, labels = data.half().to(device), labels.to(device)
            else:
                data, labels = data.to(device), labels.to(device)

            # Forward pass
            out = model(data)
            loss = crit(out, labels)

            # Accumulate loss
            total_loss += loss.item()

            # Calculate number of correct predictions
            _, preds = torch.max(out, 1)
            total_corrects += (preds == labels).sum().item()
            total_samples += labels.size(0)

    # Calculate average loss and accuracy for the entire dataset
    avg_loss = total_loss / total_samples
    avg_acc = total_corrects / total_samples

    # print(f"Epoch [{epoch}] - Test Loss: {avg_loss:.4f}, Test Accuracy: {avg_acc * 100:.2f}%")
    return avg_loss, avg_acc



def save_checkpoint(state, ckpt_path="checkpoint.pth"):
    torch.save(state, ckpt_path)
    print("Checkpoint saved")
    
# Helper function to benchmark the model
cudnn.benchmark = True
def benchmark(model, input_shape=(1024, 1, 32, 32), dtype='fp32', nwarmup=50, nruns=1000):
    input_data = torch.randn(input_shape)
    input_data = input_data.to("cuda")
    if dtype=='fp16':
        input_data = input_data.half()
        
    print("Warm up ...")
    with torch.no_grad():
        for _ in range(nwarmup):
            features = model(input_data)
    torch.cuda.synchronize()
    
    print("Start timing ...")
    timings = []
    with torch.no_grad():
        for i in range(1, nruns+1):
            start_time = time.time()
            output = model(input_data)
            torch.cuda.synchronize()
            end_time = time.time()
            timings.append(end_time - start_time)
#             if i%10==0:
#                 print('Iteration %d/%d, avg batch time %.2f ms'%(i, nruns, np.mean(timings)*1000))

    print("Input shape:", input_data.size())
    print("Output features size:", features.size())
    print('Average batch time: %.2f ms'%(np.mean(timings)*1000))
    print('Average throughput: %.2f images/second'%(input_shape[0]/np.mean(timings)))


In [10]:
# EarlyStopping class for training monitoring
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_accuracy):
        if self.best_score is None:
            self.best_score = val_accuracy
        elif val_accuracy < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_accuracy
            self.counter = 0


In [ ]:
num_epochs = 30
best_acc = 0.0
log_path = '/kaggle/working/training_log-mobilenetv2.txt'
patience = 10  # Stop training if no improvement for 5 epochs
early_stopping = EarlyStopping(patience=patience, min_delta=0.001)

# Open log file
with open(log_path, 'w') as log_file:
    log_file.write("Epoch, Train Loss, Train Acc, Val Loss, Val Acc \n")


    for epoch in range(num_epochs):
        # print('Epoch: [%5d / %5d] LR: %f' % (epoch + 1, num_epochs, lr))
    
        train_loss, train_acc = train(model, train_dataloader, criterion, optimizer, epoch)
        test_loss, test_acc = evaluate(model, val_dataloader, criterion, epoch)
        
        # Adjust learning rate
        scheduler.step(test_loss)
    
        print(f"Epoch {epoch + 1} - Training Loss: {train_loss:.4f}, Training Accuracy: {train_acc * 100:.2f}%")
        print(f"Epoch {epoch + 1} - Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc * 100:.2f}%")
        # Log metrics to file
        log_file.write(f"{epoch + 1}, {train_loss:.4f}, {train_acc:.4f}, {test_loss:.4f}, {test_acc:.4f}\n")

    
        # Save the best model
        if test_acc > best_acc:
            best_acc = test_acc
            save_checkpoint({'epoch': epoch + 1,
                             'model_state_dict': model.state_dict(),
                             'acc': test_acc,
                             'opt_state_dict': optimizer.state_dict()},
                            ckpt_path="/kaggle/working/best_mobilenetv2_ckpt")
            print(f"Best model saved at epoch {epoch + 1}")
            
        # Early stopping
        # early_stopping(test_acc)
        # if early_stopping.early_stop:
        #     print("Early stopping triggered. Stopping training.")
        #     break

print("Training complete. Best Accuracy: {:.2f}%".format(best_acc * 100))


In [ ]:
torch.save(model,'baseline_mobilenetv2.pth')

In [ ]:
import matplotlib.pyplot as plt

train_class_counts = [count for _, count in sorted(Counter(train_dataset.targets).items())]
val_class_counts = [count for _, count in sorted(Counter(val_dataset.targets).items())]

labels = sorted(Counter(train_dataset.targets).keys())

plt.bar(labels, train_class_counts, alpha=0.7, label="Training")
plt.bar(labels, val_class_counts, alpha=0.7, label="Validation")
plt.xlabel("Class Labels")
plt.ylabel("Sample Count")
plt.title("Class Distribution")
plt.legend()
plt.show()


In [ ]:
import time

def calculate_throughput(model, input_shape=(32, 3, 224, 224), nwarmup=50, nruns=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    input_data = torch.randn(input_shape).to(device)
    
    # Warm-up
    with torch.no_grad():
        for _ in range(nwarmup):
            _ = model(input_data)
    torch.cuda.synchronize()
    
    # Measure throughput
    timings = []
    with torch.no_grad():
        for _ in range(nruns):
            start_time = time.time()
            _ = model(input_data)
            torch.cuda.synchronize()
            end_time = time.time()
            timings.append(end_time - start_time)
    
    avg_batch_time = sum(timings) / nruns
    batch_size = input_shape[0]
    throughput = batch_size / avg_batch_time
    print(f"Average Throughput: {throughput:.2f} samples/second")
    return throughput


def calculate_latency(model, input_shape=(1, 3, 224, 224), nwarmup=50, nruns=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    input_data = torch.randn(input_shape).to(device)
    
    # Warm-up
    with torch.no_grad():
        for _ in range(nwarmup):
            _ = model(input_data)
    torch.cuda.synchronize()
    
    # Measure latency
    timings = []
    with torch.no_grad():
        for _ in range(nruns):
            start_time = time.time()
            _ = model(input_data)
            torch.cuda.synchronize()  # Ensure all ops are completed
            end_time = time.time()
            timings.append(end_time - start_time)
    
    avg_latency = sum(timings) / nruns
    print(f"Average Latency: {avg_latency * 1000:.2f} ms")
    return avg_latency

def calculate_model_size(model, dtype='fp32'):
    num_params = sum(p.numel() for p in model.parameters())
    
    if dtype == 'fp32':
        param_size = 4  # FP32 = 4 bytes
    elif dtype == 'fp16':
        param_size = 2  # FP16 = 2 bytes
    elif dtype == 'int8':
        param_size = 1  # INT8 = 1 byte
    elif dtype == 'int4':
        param_size = 0.5  # INT4 = 0.5 bytes
    else:
        raise ValueError("Unsupported data type")
    
    model_size = num_params * param_size / (1024 ** 2)  # Convert to MB
    print(f"Model Size: {model_size:.2f} MB")
    return model_size


In [ ]:
calculate_throughput(model, input_shape=(1, 3, 224, 224), nwarmup=50, nruns=100)
calculate_latency(model, input_shape=(1, 3, 224, 224), nwarmup=50, nruns=100)
calculate_model_size(model, dtype='fp32')

In [11]:
# Evaluate and benchmark the performance of the baseline Pytorch model (FP32 Model)
test_loss, test_acc = evaluate(model, val_dataloader, criterion, 0)
print("Mobilenetv2 TRT Baseline accuracy: {:.2f}%".format(100 * test_acc))
#print("Resnet50 TRT Baseline accuracy: {:.2f}%".format(100 * test_acc))

benchmark(model, input_shape=(1, 3, 224, 224))

Mobilenetv2 TRT Baseline accuracy: 98.88%
Warm up ...
Start timing ...
Input shape: torch.Size([1, 3, 224, 224])
Output features size: torch.Size([1, 10])
Average batch time: 4.39 ms
Average throughput: 227.54 images/second


In [ ]:
import torch
from torch import Tensor
import torch.nn as nn
from torch.hub import load_state_dict_from_url
from typing import Type, Any, Callable, Union, List, Optional
from pytorch_quantization import quant_modules
from pytorch_quantization import nn as quant_nn

__all__ = [
    'ResNet', 'resnet18', 'resnet34', 'resnet50', 'resnet101', 'resnet152', 'resnext50_32x4d', 'resnext101_32x8d',
    'wide_resnet50_2', 'wide_resnet101_2'
]

model_urls = {
    'resnet18': 'https://download.pytorch.org/models/resnet18-f37072fd.pth',
    'resnet34': 'https://download.pytorch.org/models/resnet34-b627a593.pth',
    'resnet50': 'https://download.pytorch.org/models/resnet50-0676ba61.pth',
    'resnet101': 'https://download.pytorch.org/models/resnet101-63fe2227.pth',
    'resnet152': 'https://download.pytorch.org/models/resnet152-394f9c45.pth',
    'resnext50_32x4d': 'https://download.pytorch.org/models/resnext50_32x4d-7cdf4587.pth',
    'resnext101_32x8d': 'https://download.pytorch.org/models/resnext101_32x8d-8ba56ff5.pth',
    'wide_resnet50_2': 'https://download.pytorch.org/models/wide_resnet50_2-95faca4d.pth',
    'wide_resnet101_2': 'https://download.pytorch.org/models/wide_resnet101_2-32ee1156.pth',
}


def conv3x3(in_planes: int,
            out_planes: int,
            stride: int = 1,
            groups: int = 1,
            dilation: int = 1,
            quantize: bool = False) -> nn.Conv2d:
    """3x3 convolution with padding"""
    if quantize:
        return quant_nn.QuantConv2d(in_planes,
                                    out_planes,
                                    kernel_size=3,
                                    stride=stride,
                                    padding=dilation,
                                    groups=groups,
                                    bias=False,
                                    dilation=dilation)
    else:
        return nn.Conv2d(in_planes,
                         out_planes,
                         kernel_size=3,
                         stride=stride,
                         padding=dilation,
                         groups=groups,
                         bias=False,
                         dilation=dilation)


def conv1x1(in_planes: int, out_planes: int, stride: int = 1, quantize: bool = False) -> nn.Conv2d:
    """1x1 convolution"""
    if quantize:
        return quant_nn.QuantConv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)
    else:
        return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class BasicBlock(nn.Module):
    expansion: int = 1

    def __init__(self,
                 inplanes: int,
                 planes: int,
                 stride: int = 1,
                 downsample: Optional[nn.Module] = None,
                 groups: int = 1,
                 base_width: int = 64,
                 dilation: int = 1,
                 norm_layer: Optional[Callable[..., nn.Module]] = None,
                 quantize: bool = False) -> None:
        super(BasicBlock, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        if groups != 1 or base_width != 64:
            raise ValueError('BasicBlock only supports groups=1 and base_width=64')
        if dilation > 1:
            raise NotImplementedError("Dilation > 1 not supported in BasicBlock")
        # Both self.conv1 and self.downsample layers downsample the input when stride != 1
        self.conv1 = conv3x3(inplanes, planes, stride, quantize=quantize)
        self.bn1 = norm_layer(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes, quantize=quantize)
        self.bn2 = norm_layer(planes)
        self.downsample = downsample
        self.stride = stride
        self._quantize = quantize
        if self._quantize:
            self.residual_quantizer = quant_nn.TensorQuantizer(quant_nn.QuantConv2d.default_quant_desc_input)

    def forward(self, x: Tensor) -> Tensor:
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        if self._quantize:
            out += self.residual_quantizer(identity)
        else:
            out += identity
        out = self.relu(out)

        return out


class Bottleneck(nn.Module):
    # Bottleneck in torchvision places the stride for downsampling at 3x3 convolution(self.conv2)
    # while original implementation places the stride at the first 1x1 convolution(self.conv1)
    # according to "Deep residual learning for image recognition"https://arxiv.org/abs/1512.03385.
    # This variant is also known as ResNet V1.5 and improves accuracy according to
    # https://ngc.nvidia.com/catalog/model-scripts/nvidia:resnet_50_v1_5_for_pytorch.

    expansion: int = 4

    def __init__(self,
                 inplanes: int,
                 planes: int,
                 stride: int = 1,
                 downsample: Optional[nn.Module] = None,
                 groups: int = 1,
                 base_width: int = 64,
                 dilation: int = 1,
                 norm_layer: Optional[Callable[..., nn.Module]] = None,
                 quantize: bool = False) -> None:
        super(Bottleneck, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        width = int(planes * (base_width / 64.)) * groups
        # Both self.conv2 and self.downsample layers downsample the input when stride != 1
        self.conv1 = conv1x1(inplanes, width, quantize=quantize)
        self.bn1 = norm_layer(width)
        self.conv2 = conv3x3(width, width, stride, groups, dilation, quantize=quantize)
        self.bn2 = norm_layer(width)
        self.conv3 = conv1x1(width, planes * self.expansion, quantize=quantize)
        self.bn3 = norm_layer(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride
        self._quantize = quantize
        if self._quantize:
            self.residual_quantizer = quant_nn.TensorQuantizer(quant_nn.QuantConv2d.default_quant_desc_input)

    def forward(self, x: Tensor) -> Tensor:
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        if self._quantize:
            out += self.residual_quantizer(identity)
        else:
            out += identity
        out = self.relu(out)

        return out


class ResNet(nn.Module):

    def __init__(self,
                 block: Type[Union[BasicBlock, Bottleneck]],
                 layers: List[int],
                 quantize: bool = False,
                 num_classes: int = 1000,
                 zero_init_residual: bool = False,
                 groups: int = 1,
                 width_per_group: int = 64,
                 replace_stride_with_dilation: Optional[List[bool]] = None,
                 norm_layer: Optional[Callable[..., nn.Module]] = None) -> None:
        super(ResNet, self).__init__()
        self._quantize = quantize

        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        self._norm_layer = norm_layer

        self.inplanes = 64
        self.dilation = 1
        if replace_stride_with_dilation is None:
            # each element in the tuple indicates if we should replace
            # the 2x2 stride with a dilated convolution instead
            replace_stride_with_dilation = [False, False, False]
        if len(replace_stride_with_dilation) != 3:
            raise ValueError("replace_stride_with_dilation should be None "
                             "or a 3-element tuple, got {}".format(replace_stride_with_dilation))
        self.groups = groups
        self.base_width = width_per_group

        if quantize:
            self.conv1 = quant_nn.QuantConv2d(3,
                                        self.inplanes,
                                        kernel_size=7,
                                        stride=2,
                                        padding=3,
                                        bias=False)
        else:
            self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3, bias=False)

        self.bn1 = norm_layer(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        if quantize:
            self.maxpool = quant_nn.QuantMaxPool2d(kernel_size=3, stride=2, padding=1)
        else:
            self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0], quantize=quantize)
        self.layer2 = self._make_layer(block,
                                       128,
                                       layers[1],
                                       stride=2,
                                       dilate=replace_stride_with_dilation[0],
                                       quantize=quantize)
        self.layer3 = self._make_layer(block,
                                       256,
                                       layers[2],
                                       stride=2,
                                       dilate=replace_stride_with_dilation[1],
                                       quantize=quantize)
        self.layer4 = self._make_layer(block,
                                       512,
                                       layers[3],
                                       stride=2,
                                       dilate=replace_stride_with_dilation[2],
                                       quantize=quantize)
        if quantize:
            self.avgpool = quant_nn.QuantAdaptiveAvgPool2d((1, 1))
        else:
            self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        if quantize:
            self.fc = quant_nn.QuantLinear(512 * block.expansion, num_classes)
        else:
            self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        # Zero-initialize the last BN in each residual branch,
        # so that the residual branch starts with zeros, and each residual block behaves like an identity.
        # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck):
                    nn.init.constant_(m.bn3.weight, 0)  # type: ignore[arg-type]
                elif isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)  # type: ignore[arg-type]

    def _make_layer(self,
                    block: Type[Union[BasicBlock, Bottleneck]],
                    planes: int,
                    blocks: int,
                    stride: int = 1,
                    dilate: bool = False,
                    quantize: bool = False) -> nn.Sequential:
        norm_layer = self._norm_layer
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride, quantize=quantize),
                norm_layer(planes * block.expansion),
            )

        layers = []
        layers.append(
            block(self.inplanes, planes, stride, downsample, self.groups, self.base_width, previous_dilation,
                  norm_layer, self._quantize))
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(
                block(self.inplanes,
                      planes,
                      groups=self.groups,
                      base_width=self.base_width,
                      dilation=self.dilation,
                      norm_layer=norm_layer,
                      quantize=quantize))

        return nn.Sequential(*layers)

    def _forward_impl(self, x: Tensor) -> Tensor:
        # See note [TorchScript super()]
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

    def forward(self, x: Tensor) -> Tensor:
        return self._forward_impl(x)


def _resnet(arch: str, block: Type[Union[BasicBlock, Bottleneck]], layers: List[int], pretrained: bool, progress: bool,
            quantize: bool, **kwargs: Any) -> ResNet:
    model = ResNet(block, layers, quantize, **kwargs)
    if pretrained:
        state_dict = load_state_dict_from_url(model_urls[arch], progress=progress)
        model.load_state_dict(state_dict)
    return model


def resnet18(pretrained: bool = False, progress: bool = True, quantize: bool = False, **kwargs: Any) -> ResNet:
    r"""ResNet-18 model from
    `"Deep Residual Learning for Image Recognition" <https://arxiv.org/pdf/1512.03385.pdf>`_.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
        progress (bool): If True, displays a progress bar of the download to stderr
    """
    return _resnet('resnet18', BasicBlock, [2, 2, 2, 2], pretrained, progress, quantize, **kwargs)


def resnet34(pretrained: bool = False, progress: bool = True, quantize: bool = False, **kwargs: Any) -> ResNet:
    r"""ResNet-34 model from
    `"Deep Residual Learning for Image Recognition" <https://arxiv.org/pdf/1512.03385.pdf>`_.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
        progress (bool): If True, displays a progress bar of the download to stderr
    """
    return _resnet('resnet34', BasicBlock, [3, 4, 6, 3], pretrained, progress, quantize, **kwargs)


def resnet50(pretrained: bool = False, progress: bool = True, quantize: bool = False, **kwargs: Any) -> ResNet:
    r"""ResNet-50 model from
    `"Deep Residual Learning for Image Recognition" <https://arxiv.org/pdf/1512.03385.pdf>`_.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
        progress (bool): If True, displays a progress bar of the download to stderr
    """
    return _resnet('resnet50', Bottleneck, [3, 4, 6, 3], pretrained, progress, quantize, **kwargs)


def resnet101(pretrained: bool = False, progress: bool = True, quantize: bool = False, **kwargs: Any) -> ResNet:
    r"""ResNet-101 model from
    `"Deep Residual Learning for Image Recognition" <https://arxiv.org/pdf/1512.03385.pdf>`_.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
        progress (bool): If True, displays a progress bar of the download to stderr
    """
    return _resnet('resnet101', Bottleneck, [3, 4, 23, 3], pretrained, progress, quantize, **kwargs)


def resnet152(pretrained: bool = False, progress: bool = True, quantize: bool = False, **kwargs: Any) -> ResNet:
    r"""ResNet-152 model from
    `"Deep Residual Learning for Image Recognition" <https://arxiv.org/pdf/1512.03385.pdf>`_.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
        progress (bool): If True, displays a progress bar of the download to stderr
    """
    return _resnet('resnet152', Bottleneck, [3, 8, 36, 3], pretrained, progress, quantize, **kwargs)


def resnext50_32x4d(pretrained: bool = False, progress: bool = True, quantize: bool = False, **kwargs: Any) -> ResNet:
    r"""ResNeXt-50 32x4d model from
    `"Aggregated Residual Transformation for Deep Neural Networks" <https://arxiv.org/pdf/1611.05431.pdf>`_.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
        progress (bool): If True, displays a progress bar of the download to stderr
    """
    kwargs['groups'] = 32
    kwargs['width_per_group'] = 4
    return _resnet('resnext50_32x4d', Bottleneck, [3, 4, 6, 3], pretrained, progress, quantize, **kwargs)


def resnext101_32x8d(pretrained: bool = False, progress: bool = True, quantize: bool = False, **kwargs: Any) -> ResNet:
    r"""ResNeXt-101 32x8d model from
    `"Aggregated Residual Transformation for Deep Neural Networks" <https://arxiv.org/pdf/1611.05431.pdf>`_.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
        progress (bool): If True, displays a progress bar of the download to stderr
    """
    kwargs['groups'] = 32
    kwargs['width_per_group'] = 8
    return _resnet('resnext101_32x8d', Bottleneck, [3, 4, 23, 3], pretrained, progress, quantize, **kwargs)


def wide_resnet50_2(pretrained: bool = False, progress: bool = True, quantize: bool = False, **kwargs: Any) -> ResNet:
    r"""Wide ResNet-50-2 model from
    `"Wide Residual Networks" <https://arxiv.org/pdf/1605.07146.pdf>`_.

    The model is the same as ResNet except for the bottleneck number of channels
    which is twice larger in every block. The number of channels in outer 1x1
    convolutions is the same, e.g. last block in ResNet-50 has 2048-512-2048
    channels, and in Wide ResNet-50-2 has 2048-1024-2048.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
        progress (bool): If True, displays a progress bar of the download to stderr
    """
    kwargs['width_per_group'] = 64 * 2
    return _resnet('wide_resnet50_2', Bottleneck, [3, 4, 6, 3], pretrained, progress, quantize, **kwargs)


def wide_resnet101_2(pretrained: bool = False, progress: bool = True, quantize: bool = False, **kwargs: Any) -> ResNet:
    r"""Wide ResNet-101-2 model from
    `"Wide Residual Networks" <https://arxiv.org/pdf/1605.07146.pdf>`_.

    The model is the same as ResNet except for the bottleneck number of channels
    which is twice larger in every block. The number of channels in outer 1x1
    convolutions is the same, e.g. last block in ResNet-50 has 2048-512-2048
    channels, and in Wide ResNet-50-2 has 2048-1024-2048.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
        progress (bool): If True, displays a progress bar of the download to stderr
    """
    kwargs['width_per_group'] = 64 * 2
    return _resnet('wide_resnet101_2', Bottleneck, [3, 4, 23, 3], pretrained, progress, quantize, **kwargs)


**Pytorch-Quantization Nvidia TensorRT QAT (INT8)**

In [12]:
!/usr/src/tensorrt/bin/trtexec

&&&& RUNNING TensorRT.trtexec [TensorRT v8403] # /usr/src/tensorrt/bin/trtexec
=== Model Options ===
  --uff=<file>                UFF model
  --onnx=<file>               ONNX model
  --model=<file>              Caffe model (default = no model, random weights used)
  --deploy=<file>             Caffe prototxt file
  --output=<name>[,<name>]*   Output names (it can be specified multiple times); at least one output is required for UFF and Caffe
  --uffInput=<name>,X,Y,Z     Input blob name and its dimensions (X,Y,Z=C,H,W), it can be specified multiple times; at least one is required for UFF models
  --uffNHWC                   Set if inputs are in the NHWC layout instead of NCHW (use X,Y,Z=H,W,C order in --uffInput)

=== Build Options ===
  --maxBatch                  Set max batch size and build an implicit batch engine (default = same size as --batch)
                              This option should not be used when the input model is ONNX or when dynamic shapes are provided.
  --minSh

In [ ]:
# Exporting to Onnx
dummy_input = torch.randn(1, 3, 224, 224, device='cuda')
input_names = [ "actual_input_1" ]
output_names = [ "output1" ]
torch.onnx.export(
    model,
    dummy_input,
#     "/kaggle/working/resnet50_base.onnx",
    "/kaggle/working/mobilenetv2_base.onnx",
    verbose=False,
    opset_version=13,
    do_constant_folding = False,
    input_names=input_names,
    output_names=output_names)

# Converting ONNX model to TRT
# !/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/resnet50_base.onnx --saveEngine=/kaggle/working/resnet50_base.trt
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv2_base.onnx --useSpinWait --avgRuns=100 --saveEngine=/kaggle/working/mobilenetv2_base.trt

In [ ]:
import torch

dummy_input = torch.randn(1, 3, 224, 224, device='cuda')
input_names = ["actual_input_1"]
output_names = ["output1"]

# Define dynamic axes to allow flexibility in the batch size and spatial dimensions
dynamic_axes = {
    "actual_input_1": {0: "batch_size", 2: "height", 3: "width"},  # batch size, height, width are dynamic
    "output1": {0: "batch_size"}  # output's batch size is dynamic
}

# Export to ONNX with dynamic axes
torch.onnx.export(
    model,
    dummy_input,
    "/kaggle/working/mobilenetv2_base_dynamic.onnx",
    input_names=input_names,
    output_names=output_names,
    dynamic_axes=dynamic_axes,
    verbose=False,
    opset_version=13,
    do_constant_folding=False
)

! /usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv2_base_dynamic.onnx --useSpinWait --saveEngine=/kaggle/working/mobilenetv2_base_dynamic.trt --minShapes=actual_input_1:1x3x224x224 --optShapes=actual_input_1:16x3x224x224 --maxShapes=actual_input_1:32x3x224x224 

**Half-Precision (FP16)**

In [ ]:
# Set static member of TensorQuantizer to use Pytorch’s own fake quantization functions
# quant_nn.TensorQuantizer.use_fb_fake_quant = True

# Exporting to Onnx
dummy_input = torch.randn(1, 3, 224, 224, device='cuda')
input_names = [ "actual_input_1" ]
output_names = [ "output1" ]

# Define dynamic axes to allow flexibility in the batch size and spatial dimensions
# dynamic_axes = {
#     "actual_input_1": {0: "batch_size", 2: "height", 3: "width"},  # batch size, height, width are dynamic
#     "output1": {0: "batch_size"}  # output's batch size is dynamic
# }

torch.onnx.export(
    model,
    dummy_input,
    "/kaggle/working/mobilenetv2_half.onnx",
    input_names=input_names,
    output_names=output_names,
    # dynamic_axes=dynamic_axes,  # Add dynamic_axes here
    verbose=False,
    opset_version=13,
    do_constant_folding=False
)


# Converting ONNX model to TRT
# !/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/resnet50_base_half.onnx --fp16 --saveEngine=/kaggle/working/resnet50_base_half.trt
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv2_half.onnx --fp16 --avgRuns=100 --useSpinWait --saveEngine=/kaggle/working/mobilenetv2_half.trt 

In [ ]:
# Exporting to Onnx
dummy_input = torch.randn(1, 3, 224, 224, device='cuda')
input_names = [ "actual_input_1" ]
output_names = [ "output1" ]

# Define dynamic axes to allow flexibility in the batch size and spatial dimensions
# dynamic_axes = {
#     "actual_input_1": {0: "batch_size", 2: "height", 3: "width"},  # batch size, height, width are dynamic
#     "output1": {0: "batch_size"}  # output's batch size is dynamic
# }

torch.onnx.export(
    model,
    dummy_input,
#     "/kaggle/working/resnet50_base.onnx",
    "/kaggle/working/mobilenetv2_forceint8.onnx",
    input_names=input_names,
    output_names=output_names,
    # dynamic_axes=dynamic_axes,  # Add dynamic_axes here
    verbose=False,
    opset_version=13,
    do_constant_folding=False)

# Converting ONNX model to TRT
# !/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/resnet50_base.onnx --saveEngine=/kaggle/working/resnet50_base.trt
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv2_forceint8.onnx --int8 --useSpinWait --avgRuns=100 --saveEngine=/kaggle/working/mobilenetv2_forceint8.trt

In [13]:
import copy

baseline_model_copy = copy.deepcopy(model)

In [14]:
#-----------per channel weight-----------------------------------
quant_desc_input = QuantDescriptor(num_bits=8, calib_method='entropy')
quant_nn.QuantConv2d.set_default_quant_desc_input(quant_desc_input)
quant_nn.QuantMaxPool2d.set_default_quant_desc_input(quant_desc_input)
quant_nn.QuantAdaptiveAvgPool2d.set_default_quant_desc_input(quant_desc_input)
quant_nn.QuantLinear.set_default_quant_desc_input(quant_desc_input)

# #-----------per tensor weight--------------------------------------
# quant_desc_input = QuantDescriptor(num_bits=4, calib_method='histogram', axis=None)
# quant_nn.QuantConv2d.set_default_quant_desc_input(quant_desc_input)
# quant_nn.QuantMaxPool2d.set_default_quant_desc_input(quant_desc_input)
# quant_nn.QuantConvTranspose2d.set_default_quant_desc_input(quant_desc_input)
# quant_nn.QuantLinear.set_default_quant_desc_input(quant_desc_input)

# quant_desc_weight = QuantDescriptor(num_bits=4, calib_method='histogram', axis=None)
# quant_nn.QuantConv2d.set_default_quant_desc_weight(quant_desc_weight)
# quant_nn.QuantConvTranspose2d.set_default_quant_desc_weight(quant_desc_weight)
# quant_nn.QuantLinear.set_default_quant_desc_weight(quant_desc_weight)


In [15]:
quant_modules.initialize()


In [16]:
# We define Mobilenetv2 again just like we did above
# All the regular conv, FC layers will be converted to their quantized counterparts due to quant_modules.initialize()
feature_extract = True
q_model = models.mobilenet_v2(pretrained=False)
# q_model =  models.resnet50(pretrained=True)

set_parameter_requires_grad(q_model, feature_extract)
# #----Resnet50 Model-------------------
# num_ftrs = q_model.fc.in_features
# q_model.fc = nn.Linear(num_ftrs, 10)
#----Mobilenetv2 Model-------------------
q_model.classifier[1] = nn.Linear(1280, 10)
#------------------------------------------
q_model = q_model.cuda()

In [18]:
q_model

AttributeError: 'TensorQuantizer' object has no attribute '_calibrator'

In [ ]:
# mobilenetv2_base_ckpt is the checkpoint generated from Step 2 : Training a baseline Mobilenetv2 model.
ckpt = torch.load("/kaggle/working/mobilenetv2_base_ckpt")
# ckpt = torch.load("/kaggle/working/resnet50_base_ckpt")
modified_state_dict={}
for key, val in ckpt["model_state_dict"].items():
    # Remove 'module.' from the key names
    if key.startswith('module'):
        modified_state_dict[key[7:]] = val
    else:
        modified_state_dict[key] = val

# Load the pre-trained checkpoint
model.load_state_dict(modified_state_dict)
optimizer.load_state_dict(ckpt["opt_state_dict"])

In [17]:
# mobilenetv2_base_ckpt is the checkpoint generated from Step 2 : Training a baseline Mobilenetv2 model.
ckpt = torch.load("/kaggle/working/best_mobilenetv2_ckpt")
# ckpt = torch.load("/kaggle/working/resnet50_base_ckpt")
modified_state_dict={}
for key, val in ckpt["model_state_dict"].items():
    # Remove 'module.' from the key names
    if key.startswith('module'):
        modified_state_dict[key[7:]] = val
    else:
        modified_state_dict[key] = val

# Load the pre-trained checkpoint
q_model.load_state_dict(modified_state_dict)
optimizer.load_state_dict(ckpt["opt_state_dict"])

In [ ]:
def compute_amax(model, **kwargs):
    # Load calib result
    for name, module in model.named_modules():
        if isinstance(module, quant_nn.TensorQuantizer):
            if module._calibrator is not None:
                if isinstance(module._calibrator, calib.MaxCalibrator):
                    module.load_calib_amax()
                else:
                    module.load_calib_amax(**kwargs)
            print(F"{name:40}: {module}")
    model.cuda()

def collect_stats(model, data_loader, num_batches):
    """Feed data to the network and collect statistics"""
    # Enable calibrators
    for name, module in model.named_modules():
        if isinstance(module, quant_nn.TensorQuantizer):
            if module._calibrator is not None:
                module.disable_quant()
                module.enable_calib()
            else:
                module.disable()

    # Feed data to the network for collecting stats
    for i, (image, _) in tqdm(enumerate(data_loader), total=num_batches):
        model(image.cuda())
        if i >= num_batches:
            break

    # Disable calibrators
    for name, module in model.named_modules():
        if isinstance(module, quant_nn.TensorQuantizer):
            if module._calibrator is not None:
                module.enable_quant()
                module.disable_calib()
            else:
                module.enable()

def calibrate_model(model, model_name, data_loader, num_calib_batch, calibrator, hist_percentile, out_dir):
    """
        Feed data to the network and calibrate.
        Arguments:
            model: classification model
            model_name: name to use when creating state files
            data_loader: calibration data set
            num_calib_batch: amount of calibration passes to perform
            calibrator: type of calibration to use (max/histogram)
            hist_percentile: percentiles to be used for historgram calibration
            out_dir: dir to save state files in
    """

    if num_calib_batch > 0:
        print("Calibrating model")
        with torch.no_grad():
            collect_stats(model, data_loader, num_calib_batch)

        if not calibrator == "histogram":
            compute_amax(model, method="max")
            calib_output = os.path.join(
                out_dir,
                F"{model_name}-max-{num_calib_batch*data_loader.batch_size}.pth")
            torch.save(model.state_dict(), calib_output)
        else:
            for percentile in hist_percentile:
                print(F"{percentile} percentile calibration")
                compute_amax(model, method="percentile")
                calib_output = os.path.join(
                    out_dir,
                    F"{model_name}-percentile-{percentile}-{num_calib_batch*data_loader.batch_size}.pth")
                torch.save(model.state_dict(), calib_output)

            for method in ["mse", "entropy"]:
                print(F"{method} calibration")
                compute_amax(model, method=method)
                calib_output = os.path.join(
                    out_dir,
                    F"{model_name}-{method}-{num_calib_batch*data_loader.batch_size}.pth")
                torch.save(model.state_dict(), calib_output)

In [ ]:
!mkdir calib_cache

In [ ]:
#Calibrate the model using max calibration technique.
with torch.no_grad():
    calibrate_model(
        model=q_model.eval(),
        model_name="mobilenetv2",
        data_loader=train_dataloader,
        num_calib_batch=32,
        calibrator="max",
        hist_percentile=[99.9, 99.99], 
        out_dir="/kaggle/working/calib_cache/")
"""[ 99.99, 99.999, 99.9999]"""



In [ ]:
#Evaluate model upon calibration
test_loss, test_acc = evaluate(model, val_dataloader, criterion, 0)
print("Resnet50 QAT accuracy using TensorRT: {:.2f}%".format(100 * test_acc))

In [ ]:
#Evaluate model upon calibration
test_loss, test_acc = evaluate(q_model, val_dataloader, criterion, 0)
print("Resnet50 QAT accuracy using TensorRT: {:.2f}%".format(100 * test_acc))

# benchmark(model, input_shape=(32, 3, 224, 224))

In [ ]:
# Set static member of TensorQuantizer to use Pytorch’s own fake quantization functions
quant_nn.TensorQuantizer.use_fb_fake_quant = True

# Exporting to Onnx
dummy_input = torch.randn(32, 3, 224, 224, device='cuda')
input_names = [ "actual_input_1" ]
output_names = [ "output1" ]
torch.onnx.export(
    q_model,
    dummy_input,
    "/kaggle/working/mobilenetv2_ptq.onnx",
    verbose=False,
    opset_version=11,
    do_constant_folding = False)

# Converting ONNX model to TRT
# !/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/resnet50_ptq.onnx --int8 --saveEngine=/kaggle/working/resnet50_ptq.trt
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv2_ptq.onnx --int8 --useSpinWait --saveEngine=/kaggle/working/mobilenetv2_ptq.trt

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_quantization import nn as quant_nn
from pytorch_quantization.tensor_quant import QuantDescriptor

class QuantConv2d(nn.Conv2d):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, groups=1, bias=True):
        super(QuantConv2d, self).__init__(in_channels, out_channels, kernel_size, stride, padding, dilation, groups, bias)
        # Initialize input and weight quantizers with default 8-bit precision
        self.input_quantizer = TensorQuantizer(QuantDescriptor(num_bits=8, calib_method='histogram', learn_amax=True))
        self.weight_quantizer = TensorQuantizer(QuantDescriptor(num_bits=8, calib_method='histogram', learn_amax=True))

    def forward(self, input):
        # Quantize the input and weights before applying convolution
        quant_input = self.input_quantizer(input)
        quant_weight = self.weight_quantizer(self.weight)
        return F.conv2d(quant_input, quant_weight, self.bias, self.stride, self.padding, self.dilation, self.groups)

    def update_num_bits(self, num_bits_input, num_bits_weight):
        # Update the bit-width for input and weight quantizers
        self.input_quantizer.num_bits = num_bits_input
        self.weight_quantizer.num_bits = num_bits_weight


class QuantLinear(nn.Linear):
    def __init__(self, in_features, out_features, bias=True):
        super(QuantLinear, self).__init__(in_features, out_features, bias)
        # Initialize input and weight quantizers with default 8-bit precision
        self.input_quantizer = TensorQuantizer(QuantDescriptor(num_bits=8, calib_method='histogram', learn_amax=True))
        self.weight_quantizer = TensorQuantizer(QuantDescriptor(num_bits=8, calib_method='histogram', learn_amax=True))

    def forward(self, input):
        # Quantize the input and weights before applying the linear transformation
        quant_input = self.input_quantizer(input)
        quant_weight = self.weight_quantizer(self.weight)
        return F.linear(quant_input, quant_weight, self.bias)

    def update_num_bits(self, num_bits_input, num_bits_weight):
        # Update the bit-width for input and weight quantizers
        self.input_quantizer.num_bits = num_bits_input
        self.weight_quantizer.num_bits = num_bits_weight


In [ ]:
def update_quantization_bits(model, num_bits_input, num_bits_weight):
    for module in model.children():
        # Update bits if the module has quantizers
        if hasattr(module, 'input_quantizer') and isinstance(module.input_quantizer, TensorQuantizer):
            module.input_quantizer.num_bits = num_bits_input
        if hasattr(module, 'weight_quantizer') and isinstance(module.weight_quantizer, TensorQuantizer):
            module.weight_quantizer.num_bits = num_bits_weight

        # Recurse into child modules if they exist
        if list(module.children()):  # This checks more effectively if the module has children
            update_quantization_bits(module, num_bits_input, num_bits_weight)
    return model


In [ ]:
def update_quantization_bits_with_configs(model, configs, parent_name=""):
    """
    Update the quantization bit-widths of specific layers based on a configuration dictionary.
    
    Args:
        model (nn.Module): The model to update.
        configs (dict): Dictionary where keys are layer names and values are desired bit-widths.
                        Example: {'conv1': 8, 'layer1.0.conv1': 8, 'layer1.0.conv2': 4}
        parent_name (str): The name of the parent module (for recursive tracking).
    
    Returns:
        nn.Module: The updated model with the appropriate bit-widths applied.
    """
    for name, module in model.named_children():
        # Construct the full layer name for nested layers
        layer_name = f"{parent_name}.{name}" if parent_name else name

        # Check if the layer_name exists in the configs dict
        if layer_name in configs:
            num_bits = configs[layer_name]

            # Update bits if the module has quantizers
            if hasattr(module, 'input_quantizer') and isinstance(module.input_quantizer, TensorQuantizer):
                module.input_quantizer.num_bits = 8
                print(f"Updated input quantizer for {layer_name} to {num_bits}-bit.")
            
            if hasattr(module, 'weight_quantizer') and isinstance(module.weight_quantizer, TensorQuantizer):
                module.weight_quantizer.num_bits = num_bits
                print(f"Updated weight quantizer for {layer_name} to {num_bits}-bit.")

        # Recurse into child modules if they exist
        if list(module.children()):  # Check if module has children
            update_quantization_bits_with_configs(module, configs, layer_name)
    
    return model


In [ ]:
q_model = update_quantization_bits_with_configs(q_model, configs, parent_name="")

In [ ]:
q_model

In [ ]:
#Mobilenetv2:
configs={'features.0.0': 8,
'features.1.conv.0.0': 8,
'features.1.conv.1': 8,
'features.2.conv.0.0': 4, 
'features.2.conv.1.0': 4,
'features.2.conv.2': 4,
'features.3.conv.0.0': 4, 
'features.3.conv.1.0': 4,
'features.3.conv.2': 4,
'features.4.conv.0.0': 4, 
'features.4.conv.1.0': 4,
'features.4.conv.2': 4,
'features.5.conv.0.0':4, 
'features.5.conv.1.0': 4,
'features.5.conv.2': 4,
'features.6.conv.0.0':4, 
'features.6.conv.1.0': 4,
'features.6.conv.2': 4,
'features.7.conv.0.0':4, 
'features.7.conv.1.0': 8,
'features.7.conv.2': 8,
'features.8.conv.0.0': 8,
'features.8.conv.1.0':8 ,
'features.8.conv.2': 8,
'features.9.conv.0.0':8, 
'features.9.conv.1.0': 8,
'features.9.conv.2': 4,
'features.10.conv.0.0':4, 
'features.10.conv.1.0': 4,
'features.10.conv.2': 4,
'features.11.conv.0.0':4, 
'features.11.conv.1.0': 4,
'features.11.conv.2': 4,
'features.12.conv.0.0':8, 
'features.12.conv.1.0': 8,
'features.12.conv.2': 8,
'features.13.conv.0.0':8, 
'features.13.conv.1.0': 8,
'features.13.conv.2': 8,
'features.14.conv.0.0':8, 
'features.14.conv.1.0': 8,
'features.14.conv.2': 8,
'features.15.conv.0.0':4, 
'features.15.conv.1.0': 4,
'features.15.conv.2': 8,
'features.16.conv.0.0':8, 
'features.16.conv.1.0': 8,
'features.16.conv.2': 8,
'features.17.conv.0.0':8, 
'features.17.conv.1.0': 8,
'features.17.conv.2': 8,
'features.18.0': 8,
'classifier.1': 8}

In [ ]:
#Resnet50 Model precisions
configs= {'conv1': 8,
'layer1.0.conv1': 8,
'layer1.0.conv2': 4,
'layer1.0.conv3': 4,
'layer1.0.downsample.0': 4,
'layer1.1.conv1': 8,
'layer1.1.conv2': 8,
'layer1.1.conv3': 8,
'layer1.2.conv1': 4,
'layer1.2.conv2': 4,
'layer1.2.conv3': 4,
'layer2.0.conv1': 4,
'layer2.0.conv2': 4,
'layer2.0.conv3': 4,
'layer2.0.downsample.0': 8, 
'layer2.1.conv1': 8,
'layer2.1.conv2': 8,
'layer2.1.conv3': 8,
'layer2.2.conv1': 8,
'layer2.2.conv2': 4,
'layer2.2.conv3': 4,
'layer2.3.conv1': 4,
'layer2.3.conv2': 4,
'layer2.3.conv3': 4,
'layer3.0.conv1': 4,
'layer3.0.conv2': 4,
'layer3.0.conv3': 4 ,
'layer3.0.downsample.0': 4, 
'layer3.1.conv1': 8,
'layer3.1.conv2': 8,
'layer3.1.conv3': 8,
'layer3.2.conv1': 8,
'layer3.2.conv2': 8,
'layer3.2.conv3': 8,
'layer3.3.conv1': 8,
'layer3.3.conv2': 8,
'layer3.3.conv3': 4,
'layer3.4.conv1': 4,
'layer3.4.conv2': 4,
'layer3.4.conv3': 4,
'layer3.5.conv1': 4,
'layer3.5.conv2': 4,
'layer3.5.conv3': 4,
'layer4.0.conv1': 4,
'layer4.0.conv2': 4,
'layer4.0.conv3': 8,
'layer4.0.downsample.0':8, 
'layer4.1.conv1': 8,
'layer4.1.conv2': 8,
'layer4.1.conv3': 8,
'layer4.2.conv1': 8,
'layer4.2.conv2': 8,
'layer4.2.conv3': 8,
'avgpool': 8,
'fc':8}

In [ ]:
q_model

In [ ]:
# Set static member of TensorQuantizer to use Pytorch’s own fake quantization functions
quant_nn.TensorQuantizer.use_fb_fake_quant = True

# Exporting to Onnx
dummy_input = torch.randn(32, 3, 224, 224, device='cuda')
input_names = [ "actual_input_1" ]
output_names = [ "output1" ]
torch.onnx.export(
    q_model,
    dummy_input,
    "/kaggle/working/mobilenetv2_ptq.onnx",
    verbose=False,
    opset_version=13,
    do_constant_folding = False)

# Converting ONNX model to TRT
# !/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/resnet50_ptq.onnx --int8 --saveEngine=/kaggle/working/resnet50_ptq.trt
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv2_ptq.onnx --int8 --useSpinWait --saveEngine=/kaggle/working/mobilenetv2_ptq.trt

In [ ]:
q_model.load_state_dict(torch.load('/kaggle/working/calib_cache/resnet50_minmax-percentile-99.9-1024.pth'))
q_model.eval()

# q_model_perc_ = q_model.load_state_dict(torch.load('/kaggle/working/calib_cache/mobilenet_v2-percentile-99.99-1024.pth'))
# q_model_perc_ = q_model.eval()

# q_model_mse = q_model.load_state_dict(torch.load('/kaggle/working/calib_cache/mobilenet_v2-mse-1024.pth'))
# q_model_mse = q_model.eval()

# q_model_entropy = q_model.load_state_dict(torch.load('/kaggle/working/calib_cache/mobilenet_v2-entropy-1024.pth'))
# q_model_entropy = q_model.eval()


In [ ]:
# Extract and visualize quantized weights from a specific layer, e.g., the first conv layer
# conv1_quantized_weights_int4 = q_model.conv1.weight_quantizer(q_model.conv1.weight).cpu().numpy()
# Extract and visualize quantized weights from a specific layer, e.g., the first conv layer
fp32_conv1_quantized_weights = model.conv1(model.conv1.weight).cpu().numpy()


In [ ]:
plt.hist(conv1_quantized_weights_per.flatten(), bins=100, color='blue', alpha=0.7)
plt.title('Distribution of Quantized Weights')
plt.xlabel('Weight Value')
plt.ylabel('Frequency')
plt.show()


In [ ]:
plt.hist(conv1_quantized_weights_int4.flatten(), bins=100, color='blue', alpha=0.7)
plt.title('Distribution of Quantized Weights')
plt.xlabel('Weight Value')
plt.ylabel('Frequency')
plt.show()


In [ ]:
plt.hist(fp32_conv1_quantized_weights.flatten(), bins=100, color='blue', alpha=0.7)
plt.title('Distribution of Full precision - FP32 Weights')
plt.xlabel('Weight Value')
plt.ylabel('Frequency')
plt.show()


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
baseline_model_copy.to(device)
# Example usage:
# sensitivity_results = gradient_sensitivity_analysis(model, train_dataloader)
# visualize_sensitivity(sensitivity_results)

In [ ]:
import copy

base_model = copy.deepcopy(model)

In [ ]:
# Define a small epsilon for perturbation
import matplotlib.pyplot as plt

epsilon = 1e-5
base_model.train()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model.to(device)

# Function to measure the impact of perturbations
def measure_perturbation_impact(model, dataloader, epsilon):
    sensitivity = {}
    
    inputs, labels = next(iter(dataloader))
    inputs, labels = inputs.to(device), labels.to(device)
    
    # Calculate original loss
    outputs = model(inputs)
    original_loss = criterion(outputs, labels).item()

    for name, layer in model.named_modules():
        if isinstance(layer, (nn.Conv2d, nn.Linear)):
            layer_name = name
            sensitivity[layer_name] = []
            
            for param in layer.parameters():
                # Perturb the weights
                original_weight = param.data.clone()
                param.data += epsilon * torch.sign(torch.randn_like(param.data))
                
                # Measure the loss after perturbation
                outputs = model(inputs)
                perturbed_loss = criterion(outputs, labels).item()
                
                # Calculate the impact as the absolute difference in loss
                sensitivity_impact = abs(perturbed_loss - original_loss)
                sensitivity[layer_name].append(sensitivity_impact)
                
                # Restore the original weights
                param.data = original_weight

    return sensitivity

# Perform the sensitivity analysis
sensitivity = measure_perturbation_impact(base_model, val_dataloader, epsilon)

# Visualize the results
layer_names = []
sensitivity_values = []

for layer_name, impacts in sensitivity.items():
    avg_sensitivity = sum(impacts) / len(impacts)
    layer_names.append(layer_name)
    sensitivity_values.append(avg_sensitivity)

plt.figure(figsize=(12, 8))
plt.barh(layer_names, sensitivity_values, color='pink')
plt.xlabel('Average Sensitivity Impact on Loss')
plt.ylabel('Layer Name')
plt.title('Quantization Sensitivity Analysis of Mobilenetv2 Layers')
plt.show()

In [ ]:
# Enable gradient computation for all parameters
for param in base_model.parameters():
    param.requires_grad = True


In [ ]:
def measure_normalized_gradient_sensitivity(model, dataloader, criterion):
    sensitivity_per_layer = {}
    parameter_size_per_layer = {}

    # Get a batch of inputs and labels from the dataloader
    inputs, labels = next(iter(dataloader))
    inputs, labels = inputs.to(device), labels.to(device)

    # Set model to training mode (if any dropout or BN layers)
    model.train()

    # Zero out gradients
    model.zero_grad()

    # Forward pass to calculate outputs and original loss
    outputs = model(inputs)
    original_loss = criterion(outputs, labels)
    
    # Backward pass to calculate gradients
    original_loss.backward()

    # Iterate through layers and compute gradient-based sensitivity
    for name, layer in model.named_modules():
        if isinstance(layer, (torch.nn.Conv2d, torch.nn.Linear)):  # Only focus on convolutional and linear layers
            layer_name = name
            total_sensitivity = 0
            total_params = 0

            # Loop through each parameter in the layer (weights and biases)
            for param in layer.parameters():
                if param.grad is not None:
                    # Calculate sensitivity as the sum of absolute gradients
                    sensitivity = param.grad.abs().sum().item()
                    total_sensitivity += sensitivity
                    total_params += param.numel()

            # Normalize the sensitivity by the number of parameters
            if total_params > 0:
                sensitivity_per_layer[layer_name] = total_sensitivity / total_params
            else:
                sensitivity_per_layer[layer_name] = 0

            # Store parameter size for future use
            parameter_size_per_layer[layer_name] = total_params

    return sensitivity_per_layer, parameter_size_per_layer



In [ ]:
sensitivity_per_layer, parameter_size_per_layer = measure_normalized_gradient_sensitivity(base_model, train_dataloader, criterion)

In [ ]:
sensitivity_per_layer

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming sensitivity_per_layer and parameter_size_per_layer are dictionaries with the same keys
layer_names = list(sensitivity_per_layer.keys())  # Ensure layer names are in a consistent order
sensitivity_values = [sensitivity_per_layer[name] for name in layer_names]
parameter_sizes = [parameter_size_per_layer[name] for name in layer_names]

# Normalize data if necessary (if scales are too different to be meaningfully plotted on the same graph)
sensitivity_norm = np.log10(sensitivity_values)  # Converting to a log scale if values vary widely
parameter_sizes_norm = np.log10(parameter_sizes)  # Same as above

plt.figure(figsize=(14, 8))
plt.plot(layer_names, sensitivity_norm, marker='o', label='Sensitivity')
# plt.plot(layer_names, parameter_sizes_norm, marker='x', label='Parameter Size')

plt.xlabel('Layer Names')
plt.ylabel('Sensitivity')
plt.title('Quantization sensitivity per Layer in MobileNetV2')
plt.xticks(rotation=45)  # Rotate layer names for better readability
plt.legend()
plt.grid(True)
plt.tight_layout()  # Adjust layout to make room for label rotation
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming sensitivity_per_layer and parameter_size_per_layer are dictionaries with the same keys
layer_names = list(sensitivity_per_layer.keys())  # Ensure layer names are in a consistent order
sensitivity_values = [sensitivity_per_layer[name] for name in layer_names]
parameter_sizes = [parameter_size_per_layer[name] for name in layer_names]

# Normalize data if necessary (if scales are too different to be meaningfully plotted on the same graph)
sensitivity_norm = np.log10(sensitivity_values)  # Converting to a log scale if values vary widely
parameter_sizes_norm = np.log10(parameter_sizes)  # Same as above

plt.figure(figsize=(14, 8))
plt.plot(layer_names, sensitivity_values, marker='o', label='Sensitivity')
# plt.plot(layer_names, parameter_sizes, marker='x', label='Parameter Size')

plt.xlabel('Layer Names')
plt.ylabel('Sensitivity for Quantization')
plt.title('Quantization Sensitivity per Layer in MobileNetV2')
plt.xticks(rotation=45)  # Rotate layer names for better readability
plt.yscale('log')  # Set y-axis to log scale
plt.legend()
plt.grid(True, which="both", ls="--")  # Adding gridlines for better visualization
plt.tight_layout()  # Adjust layout to make room for label rotation
plt.show()


In [ ]:
from copy import deepcopy

base_model = deepcopy(model)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model.to(device)
# Enable gradient computation for all parameters
for param in base_model.parameters():
    param.requires_grad = True

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Function to calculate channel sensitivities based on gradients
def calculate_sensitivities(model, dataloader, criterion):
    sensitivities = {}
    
    # Get a batch from the dataloader
    inputs, labels = next(iter(dataloader))
    device = next(model.parameters()).device
    inputs, labels = inputs.to(device), labels.to(device)
    
    model.train()
    model.zero_grad()
    
    # Forward and backward pass to get gradients
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    
    # Calculate sensitivity for each Conv2d layer
    for name, layer in model.named_modules():
        if isinstance(layer, nn.Conv2d):
            param = next(layer.parameters())
            if param.grad is not None:
                # Sum absolute gradients to get sensitivity per channel
                channel_sensitivity = param.grad.abs().sum(dim=(1, 2, 3)).cpu().numpy()
                sensitivities[name] = channel_sensitivity
    return sensitivities

# Function to prune and modify the model based on sensitivities
def prune_and_modify_model(model, sensitivities, pruning_threshold=0.01):
    for name, layer in model.named_modules():
        if isinstance(layer, nn.Conv2d) and name in sensitivities:
            sensitivity = sensitivities[name]
            prune_mask = torch.tensor(sensitivity >= pruning_threshold).float()
            unpruned_indices = torch.nonzero(prune_mask).flatten().tolist()
            
            # Create a new Conv2d layer with only unpruned channels
            in_channels = layer.in_channels
            out_channels = len(unpruned_indices)
            new_layer = nn.Conv2d(in_channels, out_channels, layer.kernel_size, layer.stride, layer.padding, layer.dilation)
            
            # Copy weights and biases for unpruned channels
            new_layer.weight.data = layer.weight.data[unpruned_indices, :, :, :].clone()
            if layer.bias is not None:
                new_layer.bias.data = layer.bias.data[unpruned_indices].clone()
            
            # Replace the old layer with the new pruned layer
            parent_module = dict(model.named_modules())[name.rsplit('.', 1)[0]]
            setattr(parent_module, name.split('.')[-1], new_layer)
    
    return model

# Function to calculate FLOPs for the model
def calculate_flops(model, input_shape):
    flops = 0
    device = next(model.parameters()).device
    x = torch.randn(input_shape).to(device)  # Dummy input to determine output shapes

    def compute_layer_flops(layer, x):
        if isinstance(layer, nn.Conv2d):
            out_channels, in_channels, kernel_size, _ = layer.weight.shape
            output_height, output_width = x.shape[2], x.shape[3]  # Extract height and width from the output tensor
            # FLOPs for Conv2D: output channels * output height * output width * kernel size^2 * input channels
            return out_channels * output_height * output_width * kernel_size**2 * in_channels
        elif isinstance(layer, nn.Linear):
            # FLOPs for Linear layer: output features * input features
            return layer.in_features * layer.out_features
        return 0

    # Pass through each layer to compute FLOPs
    for layer in model.modules():
        # Forward pass through Conv2D layers
        if isinstance(layer, nn.Conv2d):
            if x.dim() != 4:
                raise ValueError(f"Expected 4D input for Conv2d layer, got {x.dim()}D input of shape {x.shape}")
            x = layer(x)  # Apply Conv2d
            flops += compute_layer_flops(layer, x)
        # Forward pass through Linear layers, flattening if necessary
        elif isinstance(layer, nn.Linear):
            if x.dim() == 4:
                x = x.view(x.size(0), -1)  # Flatten only once, before the first Linear layer
            elif x.dim() != 2:
                raise ValueError(f"Expected 2D input for Linear layer, got {x.dim()}D input of shape {x.shape}")
            x = layer(x)
            flops += compute_layer_flops(layer, x)
        else:
            # Pass through non-FLOPs-contributing layers like ReLU, BatchNorm, etc.
            x = layer(x)

    return flops




# Usage example
input_shape = (1, 3, 224, 224)  # Modify based on your model input dimensions
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model.to(device)

# Calculate FLOPs for the original model
original_flops = calculate_flops(base_model, input_shape)
print('Original FLOPs:', original_flops)

# Calculate sensitivities, prune the model, and move it to device
sensitivities = calculate_sensitivities(base_model, train_dataloader, criterion)
pruned_model = prune_and_modify_model(base_model, sensitivities, pruning_threshold=0.01)
pruned_model.to(device)

# Calculate FLOPs for the pruned model
pruned_flops = calculate_flops(pruned_model, input_shape)
print('Pruned FLOPs:', pruned_flops)

# Print FLOPs reduction
flops_reduction = 100 * (original_flops - pruned_flops) / original_flops
print(f"FLOPs Reduction after Pruning: {flops_reduction:.2f}%")


In [ ]:
def build_sensitivity_profile(model, criterion, data_loader_test):
    quant_layer_names = []
    # Disable quantization for all layers initially
    for name, module in model.named_modules():
        if name.endswith("_quantizer"):
            module.disable()
            # Strip the quantizer suffix to get the base layer name
            layer_name = name.replace("._input_quantizer", "").replace("._weight_quantizer", "")
            if layer_name not in quant_layer_names:
                quant_layer_names.append(layer_name)
    
    # Evaluate the impact of quantizing each layer independently
    for i, quant_layer in enumerate(quant_layer_names):
        print("Enable quantization for layer:", quant_layer)
        for name, module in model.named_modules():
            if name.endswith("_quantizer") and quant_layer in name:
                module.enable()  # Enable quantization for the current layer
                print(f"{name:40}: {module}")
        
        # Evaluate the model with the current layer quantized
        with torch.no_grad():
            test_loss, test_acc = evaluate(model.eval(), data_loader_test, criterion, 0)
        print(f"Loss: {test_loss}, Accuracy: {test_acc}% for layer {quant_layer}")
        
        # Disable quantization again for the layer
        for name, module in model.named_modules():
            if name.endswith("_quantizer") and quant_layer in name:
                module.disable()
                print(f"{name:40}: {module}")
    
    # Print out all layers that were analyzed for sensitivity
    print("Quantization sensitivity analysis completed for layers:", quant_layer_names)


In [ ]:
torch.save(model,'/kaggle/working/mobilenetv2_modified.pth')

**Quantization Aware Training (QAT)**

In [ ]:
# Declare Learning rate
from torch.optim import lr_scheduler

# Use cross entropy loss for classification and SGD optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=1e-3, weight_decay=1e-4, momentum=0.9)

scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
# scheduler =lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.7, patience=2, verbose=True)

In [ ]:
# Finetune the QAT model for 2 epochs
num_epochs=10

for epoch in range(num_epochs):
    print('Epoch: [%5d / %5d] LR: %f' % (epoch + 1, num_epochs, lr))

    train(q_model, train_dataloader, criterion, optimizer, epoch)
    test_loss,test_acc = evaluate(q_model, val_dataloader, criterion, epoch)
    scheduler.step()
    print("Test Acc: {:.2f}%".format(100 * test_acc))
    
save_checkpoint({'epoch': epoch + 1,
                 'model_state_dict': q_model.state_dict(),
                 'acc': test_acc,
                 'opt_state_dict': optimizer.state_dict()
                 },
                ckpt_path="/kaggle/working/mobilenetv2_qat_ckpt")
#                 ckpt_path="/kaggle/working/resnet50_qat_ckpt")
#torch.save(q_model.state_dict(), "/kaggle/working/mobilenetv2_qat.pth")

In [ ]:
#Evaluate and benchmark the performance of the QAT-TRT model (TRT INT8)
test_loss, test_acc = evaluate(q_model, val_dataloader, criterion, 0)
#print("Mobilenetv2 QAT accuracy using TensorRT: {:.2f}%".format(100 * test_acc))
print("Resnet50 QAT accuracy using TensorRT: {:.2f}%".format(100 * test_acc))
# benchmark(model, input_shape=(32, 3, 224, 224))

In [ ]:
def find_matching_layer(layer_name, configs):
    """Find the longest matching layer name in the configs."""
    # Split the layer name by dots
    name_parts = layer_name.split('.')
    
    # Try matching progressively smaller prefixes
    for i in range(len(name_parts), 0, -1):
        candidate_name = '.'.join(name_parts[:i])
        if candidate_name in configs:
#             print(candidate_name)
            return candidate_name
    return None  # No match found

def estimate_mixed_precision_model_size(model, configs):
    total_size = 0
    for name, param in model.named_parameters():
        if name.endswith('.weight'):  # Only consider weights
            # Remove the '.weight' suffix
            layer_name = name.replace('.weight', '')
            
            # Find the longest matching layer name in configs
            matching_layer = find_matching_layer(layer_name, configs)
            if matching_layer:
                weight_precision = configs[matching_layer]
                # Calculate size based on weight precision
                layer_size = calculate_layer_size(param.numel(), weight_precision)
                total_size += layer_size
    
    return total_size / (1024 ** 2)  # Convert to MB

# Example usage:


# Assuming you have loaded your MobileNetV2 model
estimated_size = estimate_mixed_precision_model_size(q_model, configs)
print(f"Estimated Mixed-Precision Model Size: {estimated_size:.2f} MB")


In [ ]:
#Evaluate and benchmark the performance of the QAT-TRT model (TRT INT8)
test_loss, test_acc = evaluate(q_model, val_dataloader, criterion, 0)
#print("Mobilenetv2 QAT accuracy using TensorRT: {:.2f}%".format(100 * test_acc))
print("Resnet50 QAT accuracy using TensorRT: {:.2f}%".format(100 * test_acc))
benchmark(model, input_shape=(32, 3, 224, 224))

In [ ]:
torch.save(model,'/kaggle/working/mobilenetv2.pth')

**Benchmarking Inference**

In [ ]:
# Set static member of TensorQuantizer to use Pytorch’s own fake quantization functions
quant_nn.TensorQuantizer.use_fb_fake_quant = True

# Exporting to Onnx
dummy_input = torch.randn(32, 3, 224, 224, device='cuda')
input_names = [ "actual_input_1" ]
output_names = [ "output1" ]
torch.onnx.export(
    q_model,
    dummy_input,
    "/kaggle/working/mobilenetv2_qat.onnx",
    verbose=False,
    opset_version=13,
    do_constant_folding = False)

# Converting ONNX model to TRT
# !/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/resnet50_base_qat.onnx --int8 --saveEngine=/kaggle/working/resnet50_base_qat.trt
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv2_qat.onnx --int8 --useSpinWait --saveEngine=/kaggle/working/mobilenetv2_qat.trt

In [ ]:
# Import needed libraries and define the evaluate function

import pycuda.driver as cuda
import pycuda.autoinit
import time 
import tensorrt as trt

def evaluate_trt(engine_path, dataloader, batch_size):
    
    def predict(batch): # result gets copied into output
        # transfer input data to device
        cuda.memcpy_htod_async(d_input, batch, stream)
        # execute model
        context.execute_async_v2(bindings, stream.handle, None)
        # transfer predictions back
        cuda.memcpy_dtoh_async(output, d_output, stream)
        # syncronize threads
        stream.synchronize()
        return output
    
    with open(engine_path, 'rb') as f, trt.Runtime(trt.Logger(trt.Logger.WARNING)) as runtime, runtime.deserialize_cuda_engine(f.read()) as engine, engine.create_execution_context() as context:
        total = 0
        correct = 0
        for images, labels in val_dataloader:
            input_batch = images.numpy()
            labels = labels.numpy()
            output = np.empty([batch_size, 10], dtype = np.float32) 

            # Now allocate input and output memory, give TRT pointers (bindings) to it:
            d_input = cuda.mem_alloc(1 * input_batch.nbytes)
            d_output = cuda.mem_alloc(1 * output.nbytes)
            bindings = [int(d_input), int(d_output)]

            stream = cuda.Stream()
            preds = predict(input_batch)
            pred_labels = []
            for pred in preds:
                pred_label = (-pred).argsort()[0]
                pred_labels.append(pred_label)

            total += len(labels)
            correct += (pred_labels == labels).sum()
    
    return correct/total


In [ ]:
import numpy as np
import torch
import time
import tensorrt as trt
import pycuda.driver as cuda

def inference_trt(trt_model, dataloader, crit, epoch, dtype='fp32', batch_size_for_throughput=32):
    # Create a TensorRT context for inference
    context = trt_model.create_execution_context()

    # Allocate device memory for inputs and outputs
    input_shape = context.get_binding_shape(0)
    output_shape = context.get_binding_shape(1)
    input_size = trt.volume(input_shape) * trt.float32.itemsize
    output_size = trt.volume(output_shape) * trt.float32.itemsize

    d_input = cuda.mem_alloc(input_size)
    d_output = cuda.mem_alloc(output_size)
    
    # Allocate host buffer for output
    h_output = np.empty(output_shape, dtype=np.float32)  # Host output buffer
    
    # Lists to store timing results
    latency_times = []
    throughput_times = []
    
    total = 0
    correct = 0
    loss = 0.0
    
    for data, labels in dataloader:
        # Resize input data for throughput or latency calculation
        if batch_size_for_throughput == 1:
            # Latency mode: process one image at a time
            data = data[0:1].cuda()
        else:
            # Throughput mode: process a batch of images
            data = torch.randn(batch_size_for_throughput, *data.shape[1:]).cuda()

        labels = labels.cuda()
        
        # Transfer data to GPU
        cuda.memcpy_htod(d_input, data.cpu().numpy().astype(np.float32))
        
        # Timing for latency or throughput
        start_time = time.time()
        context.execute_v2(bindings=[int(d_input), int(d_output)])
        cuda.memcpy_dtoh(h_output, d_output)  # Copy result from device to host
        end_time = time.time()
        
        # Calculate time for latency or throughput
        if batch_size_for_throughput == 1:
            latency_times.append(end_time - start_time)
        else:
            throughput_times.append(end_time - start_time)
        
        # Calculate loss and accuracy
        output = torch.from_numpy(h_output).cuda()
        loss += crit(output, labels)
        preds = output.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    # Calculate average latency and throughput
    avg_latency = (sum(latency_times) / len(latency_times)) * 1000 if latency_times else None
    avg_batch_time = sum(throughput_times) / len(throughput_times) if throughput_times else None
    throughput = batch_size_for_throughput / avg_batch_time if avg_batch_time else None

    accuracy = correct / total
    print(f"Average Loss: {loss / total}")
    print(f"Accuracy: {accuracy * 100:.2f}%")
    if avg_latency is not None:
        print(f"Average Latency: {avg_latency:.2f} ms")
    if throughput is not None:
        print(f"Throughput: {throughput:.2f} samples/second")

    return loss / total, accuracy, avg_latency, throughput


In [ ]:
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit

# Load the TensorRT engine
TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
with open("/kaggle/working/mobilenetv2_base_dynamic.trt", "rb") as f, trt.Runtime(TRT_LOGGER) as runtime:
    trt_model_base = runtime.deserialize_cuda_engine(f.read())


In [ ]:
import numpy as np
import torch
import time
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit

def  evaluate_trt_dynamic(trt_engine_path, dataloader, criterion, batch_sizes=[1, 16, 32]):
    # Load TensorRT engine
    TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
    with open(trt_engine_path, "rb") as f, trt.Runtime(TRT_LOGGER) as runtime:
        engine = runtime.deserialize_cuda_engine(f.read())
    
    # Prepare accuracy, latency, and throughput metrics
    total_correct = {batch_size: 0 for batch_size in batch_sizes}
    total_samples = {batch_size: 0 for batch_size in batch_sizes}
    latencies = {batch_size: [] for batch_size in batch_sizes}
    throughputs = {}
    
    # Evaluate for each batch size
    for batch_size in batch_sizes:
        print(f"\nEvaluating for batch size: {batch_size}")

        # Create context and allocate memory based on batch size
        context = engine.create_execution_context()
        input_shape = (batch_size, 3, 224, 224)  # Adjust as needed
        context.set_binding_shape(0, input_shape)
        
        # Allocate GPU memory for inputs and outputs
        input_size = trt.volume(input_shape) * trt.float32.itemsize
        output_shape = context.get_binding_shape(1)
        output_size = trt.volume(output_shape) * trt.float32.itemsize

        d_input = cuda.mem_alloc(input_size)
        d_output = cuda.mem_alloc(output_size)
        h_output = np.empty(output_shape, dtype=np.float32)  # Host output buffer
        
        # Initialize timing
        total_time = 0.0
        total_batches = 0

        for i, (data, labels) in enumerate(dataloader):
            if data.size(0) < batch_size:
                break  # Ensure complete batches for consistency

            data = data[0:batch_size].cuda()  # Select batch size
            labels = labels[0:batch_size].cuda()

            # Transfer data to the GPU
            cuda.memcpy_htod(d_input, data.cpu().numpy().astype(np.float32))

            # Measure latency
            start_time = time.time()
            context.execute_v2(bindings=[int(d_input), int(d_output)])
            cuda.memcpy_dtoh(h_output, d_output)
            end_time = time.time()

            latency = end_time - start_time
            latencies[batch_size].append(latency)
            total_time += latency
            total_batches += 1

            # Convert output to PyTorch tensor for accuracy calculation
            output = torch.from_numpy(h_output).cuda()
            loss = criterion(output, labels)
            preds = output.argmax(dim=1)
            total_correct[batch_size] += (preds == labels).sum().item()
            total_samples[batch_size] += labels.size(0)

        # Throughput calculation
        avg_latency = np.mean(latencies[batch_size]) * 1000  # in ms
        throughput = (total_batches * batch_size) / total_time  # samples per second
        throughputs[batch_size] = throughput

        # Print results
        accuracy = total_correct[batch_size] / total_samples[batch_size]
        print(f"Batch Size: {batch_size}")
        print(f"Accuracy: {accuracy * 100:.2f}%")
        print(f"Average Latency: {avg_latency:.2f} ms")
        print(f"Throughput: {throughput:.2f} samples/second")

    return total_correct, latencies, throughputs


In [ ]:
trt_engine_path = "/kaggle/working/mobilenetv2_half.trt"
total_correct, latencies, throughputs = evaluate_trt_dynamic(
    trt_engine_path, val_dataloader, criterion, batch_sizes=[16]
)


In [ ]:
trt_engine_path = "/kaggle/working/mobilenetv2_base_dynamic.trt"
total_correct, latencies, throughputs = evaluate_trt_dynamic(
    trt_engine_path, val_dataloader, criterion, batch_sizes=[16]
)


In [ ]:
inference_trt(trt_model, val_dataloader, criterion, epoch=0, dtype='fp32', batch_size_for_throughput=32)


In [ ]:
# Calculate the size of the file
file_size = os.path.getsize('/kaggle/working/resnet50_base.trt') / (1024 * 1024)  # Convert from bytes to megabytes
print(f'Model size: {file_size:.2f} MB')
file_size = os.path.getsize('/kaggle/working/resnet50_ptq.trt') / (1024 * 1024)  # Convert from bytes to megabytes
print(f'Model size: {file_size:.2f} MB')
file_size = os.path.getsize('/kaggle/working/resnet50_base_qat.trt') / (1024 * 1024)  # Convert from bytes to megabytes
print(f'Model size: {file_size:.2f} MB')
file_size = os.path.getsize('/kaggle/working/resnet50_base_half.trt') / (1024 * 1024)  # Convert from bytes to megabytes
print(f'Model size: {file_size:.2f} MB')


In [ ]:
# Calculate the size of the file
file_size = os.path.getsize('/kaggle/working/mobilenetv2_base.trt') / (1024 * 1024)  # Convert from bytes to megabytes
print(f'Model size: {file_size:.2f} MB')
file_size = os.path.getsize('/kaggle/working/mobilenetv2_half.trt') / (1024 * 1024)  # Convert from bytes to megabytes
print(f'Model size: {file_size:.2f} MB')
file_size = os.path.getsize('/kaggle/working/mobilenetv2_ptq.trt') / (1024 * 1024)  # Convert from bytes to megabytes
print(f'Model size: {file_size:.2f} MB')
file_size = os.path.getsize('/kaggle/working/resnet50_base_qat.trt') / (1024 * 1024)  # Convert from bytes to megabytes
print(f'Model size: {file_size:.2f} MB')


In [ ]:
# Evaluate and benchmark the performance of the baseline TRT model (TRT FP32 Model)
batch_size = 32
test_acc = evaluate_trt("/kaggle/working/mobilenetv2_base.trt", val_dataloader, batch_size)
print("Mobilenetv2 TRT Baseline accuracy: {:.2f}%".format(100 * test_acc))
############################################################################################
test_acc = evaluate_trt("/kaggle/working/mobilenetv2_half.trt", val_dataloader, batch_size)
print("Mobilenetv2 TRT FP16 accuracy: {:.2f}%".format(100 * test_acc))
############################################################################################
# test_acc = evaluate_trt("/kaggle/working/mobilenetv2_ptq.trt", val_dataloader, batch_size)
# print("Mobilenetv2 TRT PTQ accuracy: {:.2f}%".format(100 * test_acc))
# ###############################################################################################
# test_acc = evaluate_trt("/kaggle/working/resnet50_base_qat.trt", val_dataloader, batch_size)
# print("Mobilenetv2 TRT QAT accuracy: {:.2f}%".format(100 * test_acc))